# Prática 1 — Memória de agentes


In [1]:
# 0. Instalação reproduzível (rode uma vez por ambiente/kernel)
from pathlib import Path
import subprocess, sys

# Versões fixadas: o notebook não depende de um requirements externo.
if subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True).returncode:
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--only-binary=:all:",
    "openai==3.57.0", "langgraph==1.2.24", "python-dotenv==1.2.4",
    "oracledb==4.0.2", "oci==2.192.0", "mem0ai==2.2.1",
    "litellm==1.110.3", "urllib3==2.8.0", "pillow==13.0.0",
    "exceptiongroup==1.3.2", "pandas==3.1.1",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--only-binary=:all:", "--no-deps",
    "oracleagentmemory==26.8.0",
])
print("Bibliotecas instaladas. Se foi a primeira execução, reinicie o kernel.")

Bibliotecas instaladas. Se foi a primeira execução, reinicie o kernel.


In [11]:
# 1. Carrega o .env e prepara o wallet mTLS sem expor segredos
from pathlib import Path
import os, tempfile, uuid, zipfile
from dotenv import load_dotenv

ROOT = Path.cwd() if (Path.cwd() / ".env").is_file() else Path.cwd().parent
load_dotenv(ROOT / ".env", override=True)  # o .env do projeto vence variáveis antigas do kernel
os.environ["MEM0_TELEMETRY"] = "false"  # precisa ocorrer antes do primeiro import do Mem0

# Se o pacote já foi importado neste kernel, atualiza também a constante em memória.
import sys
if "mem0.memory.main" in sys.modules:
    sys.modules["mem0.memory.main"].MEM0_TELEMETRY = False

# Limpa recursos de uma execução anterior no mesmo kernel.
previous_mem0 = globals().pop("mem0", None)
if previous_mem0 is not None:
    for store_name in ("vector_store", "_telemetry_vector_store"):
        qdrant_client = getattr(getattr(previous_mem0, store_name, None), "client", None)
        if qdrant_client is not None:
            qdrant_client.close()
    previous_mem0.close()

for resource_name in ("oracle_memory", "connection"):
    resource = globals().pop(resource_name, None)
    close = getattr(resource, "close", None)
    if callable(close):
        try: close()
        except Exception: pass

for temp_name in ("mem0_temp", "wallet_temp"):
    temp_resource = globals().pop(temp_name, None)
    if temp_resource is not None:
        try: temp_resource.cleanup()
        except Exception: pass

RUN_ID = uuid.uuid4().hex[:10]

required = [
    "OCI_GENAI_API_KEY", "OCI_GENAI_BASE_URL", "OCI_GENAI_MODEL",
    "OCI_EMBEDDING_MODEL", "OCI_EMBEDDING_REGION",
    "OCI_EMBEDDING_COMPARTMENT_ID", "DB_USER", "DB_PASSWORD", "DB_DSN",
]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise ValueError("Preencha no .env: " + ", ".join(missing))

wallet_temp = None
wallet_dir = Path(os.getenv("DB_WALLET_DIR", ""))
wallet_ready = (
    wallet_dir.is_dir()
    and (wallet_dir / "tnsnames.ora").is_file()
    and (wallet_dir / "ewallet.pem").is_file()
)
if os.getenv("DB_WALLET_ZIP") and not wallet_ready:
    wallet_temp = tempfile.TemporaryDirectory(prefix="cafe-ia-wallet-")
    with zipfile.ZipFile(Path(os.environ["DB_WALLET_ZIP"]).expanduser()) as archive:
        for filename in ("tnsnames.ora", "ewallet.pem"):
            item = next(x for x in archive.infolist() if Path(x.filename).name == filename)
            (Path(wallet_temp.name) / filename).write_bytes(archive.read(item))
    os.environ["DB_WALLET_DIR"] = wallet_temp.name
    os.environ.setdefault("DB_WALLET_PASSWORD", os.environ["DB_PASSWORD"])

print(f"Sessão {RUN_ID} pronta.")

Sessão 538556faee pronta.


## 2. Conversa usada nas três estratégias

In [3]:
TURNS = [
    "Olá, meu nome é Marina e trabalho na empresa Aurora.",
    "Nosso contrato é o XPTO-4821.",
    "Prefiro receber atualizações por e-mail.",
    "Estamos com lentidão no painel desde ontem.",
    "O problema acontece principalmente no fechamento mensal.",
    "Nossa região é São Paulo.",
    "A prioridade desse chamado é alta.",
    "Já reiniciamos a integração duas vezes.",
    "Não podemos fazer uma parada durante o horário comercial.",
    "Pode registrar que a janela ideal é depois das 20h?",
    "Também inclua o time financeiro nas atualizações.",
    "Resuma meus dados, o problema e a janela de atendimento.",
]
TURNS

['Olá, meu nome é Marina e trabalho na empresa Aurora.',
 'Nosso contrato é o XPTO-4821.',
 'Prefiro receber atualizações por e-mail.',
 'Estamos com lentidão no painel desde ontem.',
 'O problema acontece principalmente no fechamento mensal.',
 'Nossa região é São Paulo.',
 'A prioridade desse chamado é alta.',
 'Já reiniciamos a integração duas vezes.',
 'Não podemos fazer uma parada durante o horário comercial.',
 'Pode registrar que a janela ideal é depois das 20h?',
 'Também inclua o time financeiro nas atualizações.',
 'Resuma meus dados, o problema e a janela de atendimento.']

## 3. Mesmo LLM e mesmo agente LangGraph para todos

In [4]:
from dataclasses import dataclass
from time import perf_counter
from typing import TypedDict
from openai import OpenAI
from langgraph.graph import START, END, StateGraph

MODEL = os.environ["OCI_GENAI_MODEL"]
client = OpenAI(
    api_key=os.environ["OCI_GENAI_API_KEY"],
    base_url=os.environ["OCI_GENAI_BASE_URL"],  # endpoint OpenAI-compatible: .../openai/v1
    timeout=90,
)

@dataclass
class LLMResult:
    text: str
    input_tokens: int
    output_tokens: int
    latency_s: float

def call_llm(prompt, system="Você é o atendente da XPTO. Responda de forma curta e útil."):
    started = perf_counter()
    response = client.responses.create(model=MODEL, instructions=system, input=prompt)
    usage = response.usage
    return LLMResult(response.output_text, usage.input_tokens, usage.output_tokens,
                     perf_counter() - started)

class AgentState(TypedDict):
    message: str
    memories: list[str]
    result: LLMResult | None

def answer(state: AgentState):
    context = "\n".join(f"- {x}" for x in state["memories"]) or "- nenhuma"
    prompt = f"MEMÓRIAS:\n{context}\n\nMENSAGEM ATUAL:\n{state['message']}"
    return {**state, "result": call_llm(prompt)}

graph = StateGraph(AgentState)
graph.add_node("atender", answer)
graph.add_edge(START, "atender")
graph.add_edge("atender", END)
agent = graph.compile()

def run_strategy(name, recall, remember):
    metric = {"estratégia": name, "tokens_entrada": 0, "tokens_saída": 0,
              "leitura_s": 0.0, "escrita_s": 0.0, "agente_s": 0.0,
              "itens_recuperados": 0, "resposta_final": ""}
    for message in TURNS:
        started = perf_counter(); memories = recall(message)
        metric["leitura_s"] += perf_counter() - started
        metric["itens_recuperados"] += len(memories)
        state = agent.invoke({"message": message, "memories": memories, "result": None})
        result = state["result"]
        metric["tokens_entrada"] += result.input_tokens
        metric["tokens_saída"] += result.output_tokens
        metric["agente_s"] += result.latency_s
        metric["resposta_final"] = result.text
        started = perf_counter(); remember(message, result.text)
        metric["escrita_s"] += perf_counter() - started
    metric["tempo_total_s"] = metric["leitura_s"] + metric["escrita_s"] + metric["agente_s"]
    return metric

## 4. Estratégia 1 — histórico inteiro no prompt

In [5]:
history = []

def history_recall(_query):
    return history.copy()

def history_remember(user_message, assistant_message):
    history.extend([f"Usuário: {user_message}", f"Agente: {assistant_message}"])

baseline = run_strategy("Histórico inteiro", history_recall, history_remember)
baseline

{'estratégia': 'Histórico inteiro',
 'tokens_entrada': 3690,
 'tokens_saída': 717,
 'leitura_s': 1.2300093658268452e-05,
 'escrita_s': 3.020011354237795e-05,
 'agente_s': 20.489341400098056,
 'itens_recuperados': 132,
 'resposta_final': '- **Cliente:** Marina, empresa Aurora  \n- **Contrato:** XPTO-4821  \n- **Região:** São Paulo  \n- **Problema:** Lentidão no painel desde ontem, principalmente no fechamento mensal, mesmo após reiniciar a integração duas vezes  \n- **Prioridade:** Alta  \n- **Janela de atendimento:** Após as 20h, sem parada durante o horário comercial  \n- **Atualizações:** Por e-mail, incluindo o time financeiro',
 'tempo_total_s': 20.489383900305256}

## 5. Arquitetura do Mem0

```mermaid
flowchart LR
    U[Mensagem do usuário] --> A[Agente LangGraph]

    subgraph APP["Aplicação / VM — responsabilidade do time"]
        A --> M[Mem0]
        M --> Q[(Qdrant local<br/>vetores)]
        M --> S[(SQLite local<br/>histórico)]
    end

    M --> L[LLM de extração<br/>OCI OpenAI-compatible]
    M --> E[OCI Generative AI<br/>Embedding]
    E --> Q
    Q --> M --> A

    classDef owned fill:#fff0e6,stroke:#d35400,stroke-width:2px;
    class Q,S owned;
```

No Mem0, a aplicação precisa instalar, versionar, proteger e operar o banco vetorial e o armazenamento de histórico.

## 6. Estratégia 2 — configuração completa do Mem0

In [6]:
# Mem0 não possui provider OCI nativo: estes dois adaptadores usam OCI
# mantendo a API pública Memory.from_config().
from pathlib import Path
from typing import Literal
import oci
from litellm import embedding
from mem0.embeddings.base import EmbeddingBase
from mem0.llms.openai import OpenAILLM

class OCIEmbedding(EmbeddingBase):
    def __init__(self, config=None):
        super().__init__(config)
        auth = os.getenv("OCI_AUTH_MODE", "config_file").lower()
        if auth == "instance_principal":
            self.signer = oci.auth.signers.InstancePrincipalsSecurityTokenSigner()
            config_region = None
        else:
            cfg = oci.config.from_file(
                os.path.expanduser(os.getenv("OCI_CONFIG_FILE", "~/.oci/config")),
                os.getenv("OCI_CONFIG_PROFILE", "DEFAULT"),
            )
            self.signer = oci.signer.Signer(
                tenancy=cfg["tenancy"], user=cfg["user"], fingerprint=cfg["fingerprint"],
                private_key_file_location=cfg["key_file"], pass_phrase=cfg.get("pass_phrase"),
            )
            config_region = cfg.get("region")
        self.model = os.environ["OCI_EMBEDDING_MODEL"]
        self.region = os.getenv("OCI_EMBEDDING_REGION", config_region)
        self.compartment = os.environ["OCI_EMBEDDING_COMPARTMENT_ID"]
        self.dimensions = int(os.getenv("OCI_EMBEDDING_DIMENSIONS", "1536"))

    def embed(self, text, memory_action=None):
        return self.embed_batch([text], memory_action or "add")[0]

    def embed_batch(self, texts, memory_action="add"):
        response = embedding(
            model=f"oci/{self.model}", input=texts, oci_signer=self.signer,
            oci_compartment_id=self.compartment, oci_region=self.region,
            input_type="search_query" if memory_action == "search" else "search_document",
            dimensions=self.dimensions,
        )
        return [list(item["embedding"]) for item in response.data]

class OCILLM(OpenAILLM):
    def _uses_max_completion_tokens(self, model):
        return True  # OCI rejeita max_tokens para o modelo usado

    def _get_supported_params(self, **kwargs):
        params = super()._get_supported_params(**kwargs)
        params.pop("temperature", None); params.pop("top_p", None)
        return params

In [7]:
os.environ["MEM0_TELEMETRY"] = "false"
from mem0 import Memory
import mem0.memory.main as mem0_main
from mem0.utils.factory import EmbedderFactory, LlmFactory
import uuid
mem0_main.MEM0_TELEMETRY = False
dimensions = int(os.getenv("OCI_EMBEDDING_DIMENSIONS", "1536"))

# Reexecutar a célula não pode deixar o Qdrant anterior bloqueando a pasta.
previous_mem0 = globals().pop("mem0", None)
if previous_mem0 is not None:
    for store_name in ("vector_store", "_telemetry_vector_store"):
        qdrant_client = getattr(getattr(previous_mem0, store_name, None), "client", None)
        if qdrant_client is not None:
            qdrant_client.close()
    previous_mem0.close()

previous_mem0_temp = globals().pop("mem0_temp", None)
if previous_mem0_temp is not None:
    previous_mem0_temp.cleanup()

# Cada execução usa armazenamento local isolado e descartável.
mem0_temp = tempfile.TemporaryDirectory(prefix="cafe-ia-mem0-")
data_dir = Path(mem0_temp.name)

# Registra os adaptadores definidos acima.
EmbedderFactory.provider_to_class["openai"] = f"{OCIEmbedding.__module__}.OCIEmbedding"
_, config_class = LlmFactory.provider_to_class["openai"]
LlmFactory.provider_to_class["openai"] = (f"{OCILLM.__module__}.OCILLM", config_class)

mem0_config = {
    "llm": {"provider": "openai", "config": {
        "model": MODEL, "api_key": os.environ["OCI_GENAI_API_KEY"],
        "openai_base_url": os.environ["OCI_GENAI_BASE_URL"],
    }},
    "embedder": {"provider": "openai", "config": {
        "model": os.environ["OCI_EMBEDDING_MODEL"], "embedding_dims": dimensions,
    }},
    "vector_store": {"provider": "qdrant", "config": {
        "collection_name": os.getenv("MEM0_COLLECTION_NAME", "cafe_ia_memories"),
        "path": str(data_dir / "qdrant"), "embedding_model_dims": dimensions,
    }},
    "history_db_path": str(data_dir / "history.db"),
}
mem0 = Memory.from_config(mem0_config)
mem0_user = "demo-" + uuid.uuid4().hex[:10]

def mem0_recall(query):
    response = mem0.search(query=query, filters={"user_id": mem0_user}, top_k=5)
    items = response.get("results", response) if isinstance(response, dict) else response
    return [str(x.get("memory") or x.get("content")) for x in items or []
            if isinstance(x, dict) and (x.get("memory") or x.get("content"))]

def mem0_remember(user_message, assistant_message):
    mem0.add([{"role": "user", "content": user_message},
              {"role": "assistant", "content": assistant_message}], user_id=mem0_user)

mem0_result = run_strategy("Mem0", mem0_recall, mem0_remember)
mem0_result

Failed to load spaCy lemma model: spaCy is not installed. Install it with: pip install mem0ai[nlp]
Failed to load spaCy full model: spaCy is not installed. Install it with: pip install mem0ai[nlp]
fastembed not installed - BM25 keyword search disabled. Install it with: pip install "mem0ai[extras]"


{'estratégia': 'Mem0',
 'tokens_entrada': 1677,
 'tokens_saída': 933,
 'leitura_s': 2.5284406999126077,
 'escrita_s': 36.6636238998035,
 'agente_s': 23.426418800256215,
 'itens_recuperados': 46,
 'resposta_final': '- **Contrato:** XPTO-4821  \n- **Problema:** lentidão no painel, principalmente durante o fechamento mensal. A integração já foi reiniciada duas vezes, sem sucesso. Ainda é necessário confirmar se afeta todos os usuários e se há mensagem de erro.  \n- **Janela de atendimento:** fora do horário comercial, preferencialmente após as 20h; horário exato ainda será definido pela Aurora.',
 'tempo_total_s': 62.61848339997232}

## 7. Arquitetura do Oracle AI Agent Memory

```mermaid
flowchart LR
    U[Mensagem do usuário] --> A[Agente LangGraph]

    subgraph APP["Aplicação"]
        A --> SDK[Oracle AI Agent Memory SDK]
    end

    SDK --> L[LLM de extração<br/>OCI OpenAI-compatible]
    SDK --> E[OCI Generative AI<br/>Embedding]

    subgraph ADB["Autonomous AI Database — persistência e busca gerenciadas"]
        T[(Threads e mensagens)]
        V[(Vetores e índices)]
        F[(Memórias, fatos<br/>e preferências)]
    end

    SDK --> T
    SDK --> V
    SDK --> F
    E --> V
    T --> SDK
    V --> SDK
    F --> SDK
    SDK --> A

    classDef managed fill:#eaf7ee,stroke:#16883e,stroke-width:2px;
    class T,V,F managed;
```

Com o Agent Memory, threads, memórias, vetores e índices ficam no Autonomous AI Database

## 8. Estratégia 3 — configuração completa do Oracle AI Agent Memory

In [8]:
# Conexão mTLS: encontra no wallet o alias correspondente ao SERVICE_NAME.
import re, asyncio
import numpy as np
import oracledb
from mem0.configs.embeddings.base import BaseEmbedderConfig
from oracleagentmemory.apis.embedders.embedder import IEmbedder

def db_connect_kwargs():
    dsn = os.environ["DB_DSN"]
    kwargs = {"user": os.environ["DB_USER"], "password": os.environ["DB_PASSWORD"], "dsn": dsn}
    wallet = Path(os.getenv("DB_WALLET_DIR", "/etc/cafe-ia/wallet"))
    if os.getenv("DB_WALLET_ZIP") and not (
        (wallet / "tnsnames.ora").is_file() and (wallet / "ewallet.pem").is_file()
    ):
        raise RuntimeError("Wallet temporário ausente. Execute novamente a célula 1 do .env.")
    if (wallet / "tnsnames.ora").is_file() and (wallet / "ewallet.pem").is_file():
        service = re.search(r"\(SERVICE_NAME\s*=\s*([^\)]+)\)", dsn, re.I).group(1).strip().lower()
        text = (wallet / "tnsnames.ora").read_text(encoding="utf-8")
        aliases = list(re.finditer(r"^\s*([A-Za-z0-9_.-]+)\s*=", text, re.M))
        alias = next(m.group(1) for i, m in enumerate(aliases)
                     if service in text[m.start():(aliases[i+1].start() if i+1 < len(aliases) else len(text))].lower())
        kwargs.update(dsn=os.getenv("DB_WALLET_DSN", alias), config_dir=str(wallet),
                      wallet_location=str(wallet),
                      wallet_password=os.getenv("DB_WALLET_PASSWORD", os.environ["DB_PASSWORD"]))
    return kwargs

class OCIAgentMemoryEmbedder(IEmbedder):
    def __init__(self):
        self._dimensions = int(os.getenv("OCI_EMBEDDING_DIMENSIONS", "1536"))
        self._delegate = OCIEmbedding(BaseEmbedderConfig(
            model=os.environ["OCI_EMBEDDING_MODEL"], embedding_dims=self._dimensions))
    @property
    def embedding_dimension(self): return self._dimensions
    @property
    def max_input_tokens(self): return int(os.getenv("OCI_EMBEDDING_MAX_INPUT_TOKENS", "512"))
    def embed(self, texts, *, is_query=False):
        action = "search" if is_query else "add"
        return np.asarray(self._delegate.embed_batch(texts, action), dtype=np.float32)
    async def embed_async(self, texts, *, is_query=False):
        return await asyncio.to_thread(self.embed, texts, is_query=is_query)

In [9]:
from oracleagentmemory.core import (
    MemoryExtractionConfig, OracleAgentMemory, SchemaPolicy, SearchStrategy,
)
from oracleagentmemory.core.extractors.memoryextractionconfig import MemoryExtractionMode
from oracleagentmemory.core.llms.llm import Llm

provider_model = MODEL if "/" in MODEL else f"openai/{MODEL}"

# Proteção para reexecução direta desta célula sem passar pela célula 1.
previous_oracle_memory = globals().pop("oracle_memory", None)
if previous_oracle_memory is not None:
    try: previous_oracle_memory.close()
    except Exception: pass
previous_connection = globals().pop("connection", None)
if previous_connection is not None:
    try: previous_connection.close()
    except Exception: pass

connection = oracledb.connect(**db_connect_kwargs())

extraction = MemoryExtractionConfig(
    extract_memories=True,
    extraction_mode=MemoryExtractionMode.INLINE,  # extrai durante add_messages
    memory_extraction_frequency=4,                # a cada quatro mensagens
    memory_extraction_window=4,                   # olha os quatro eventos recentes
    memory_extraction_token_limit=2000,
    enable_context_summary=False,
    memory_extraction_custom_instructions=(
        "Extraia apenas fatos duráveis: identidade, empresa, contrato, preferências, "
        "problema, prioridade, restrições e janela de atendimento. Evite redundâncias."
    ),
)

oracle_memory = OracleAgentMemory(
    connection=connection,
    embedder=OCIAgentMemoryEmbedder(),
    llm=Llm(model=provider_model, api_base=os.environ["OCI_GENAI_BASE_URL"],
            api_key=os.environ["OCI_GENAI_API_KEY"]),
    memory_store_id=os.getenv("ORACLE_VECTOR_MEMORY_STORE_ID", "CAFEIA_VECTOR"),
    schema_policy=getattr(SchemaPolicy, os.getenv(
        "ORACLE_MEMORY_SCHEMA_POLICY", "CREATE_IF_NECESSARY").upper()),
    search_strategy=SearchStrategy.VECTOR,
    memory_extraction_config=extraction,
)
oracle_thread = oracle_memory.create_thread(
    thread_id="thread-" + uuid.uuid4().hex, user_id=mem0_user, agent_id="xpto-support")

def oracle_recall(query):
    results = oracle_thread.search(
        query, max_results=5, token_budget=500, exact_thread_match=True,
        record_types=["message", "memory", "fact", "preference", "guideline"],
    )
    texts = []
    for item in results:
        content = getattr(item, "content", None) or getattr(getattr(item, "record", None), "content", None)
        if content: texts.append(str(content))
    return texts

def oracle_remember(user_message, _assistant_message):
    # Só a fala do usuário contém fatos novos sobre o cliente.
    oracle_thread.add_messages([{"role": "user", "content": user_message}])

oracle_result = run_strategy("Oracle AI Agent Memory", oracle_recall, oracle_remember)
oracle_result

oracleagentmemory/core/thread.py:4750: UserWarning: You are calling an asynchronous method in a synchronous method from an asynchronous context. This is highly discouraged because it can lead to deadlocks. Please use the asynchronous method equivalent: 
oracleagentmemory/core/thread.py:1092: UserWarning: You are calling an asynchronous method in a synchronous method from an asynchronous context. This is highly discouraged because it can lead to deadlocks. Please use the asynchronous method equivalent: add_messages_async


{'estratégia': 'Oracle AI Agent Memory',
 'tokens_entrada': 1145,
 'tokens_saída': 1015,
 'leitura_s': 2.3140432999935,
 'escrita_s': 36.26484770036768,
 'agente_s': 21.400767800048925,
 'itens_recuperados': 46,
 'resposta_final': '- **Empresa:** Aurora  \n- **Problema:** Lentidão no painel desde 25/11/2026, principalmente no fechamento mensal. A integração já foi reiniciada duas vezes.  \n- **Prioridade:** Alta  \n- **Janela ideal de atendimento:** Após as 20h.',
 'tempo_total_s': 59.97965880041011}

## 9. Comparativo final

In [ ]:
import pandas as pd

results = pd.DataFrame([baseline, mem0_result, oracle_result])
base_tokens = results.loc[0, "tokens_entrada"]
base_time = results.loc[0, "tempo_total_s"]
results["dif_tokens_vs_histórico_%"] = (results["tokens_entrada"] / base_tokens - 1) * 100
results["dif_tempo_vs_histórico_%"] = (results["tempo_total_s"] / base_time - 1) * 100
display(results.round(2))


,estratégia,tokens_entrada,tokens_saída,leitura_s,escrita_s,agente_s,itens_recuperados,resposta_final,tempo_total_s,dif_tokens_vs_histórico_%,dif_tempo_vs_histórico_%
0,Histórico inteiro,3690,717,0.00,0.00,20.49,132,"- **Cliente:** Marina, empresa Aurora \n- **C...",20.49,0.00,0.00
1,Mem0,1677,933,2.53,36.66,23.43,46,- **Contrato:** XPTO-4821 \n- **Problema:** l...,62.62,-54.55,205.61
2,Oracle AI Agent Memory,1145,1015,2.31,36.26,21.40,46,- **Empresa:** Aurora \n- **Problema:** Lenti...,59.98,-68.97,192.74


Percentual negativo = redução; positivo = aumento.
